# 07 · Does the screen work? (plan 7g, experiments A and C)

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 29 Oct 2026 · **Needs:** `results/01_inventory/` to `results/06_image_posthoc/` · **Plan:** section
**7g** (approved 29 Oct 2026, before any data for it was downloaded). Its results are labeled as added after the first
results; nothing in notebooks 01–06 changes.

**What this notebook does, in plain words**

Notebook 03 flagged 4 of 26 tasks from the scans' "label cards" alone, without images. Notebooks 05–06 showed that for
flagged tasks a CT model can predict the class even with the patient removed from the scan. This notebook asks:
**A. Does the label-card screen work?** It runs the same image tests (full image, background only, silhouette) on
   **all 15 CT tasks the screen did not flag**. Over all 19 CT tasks it then asks whether the label-card AUC predicts the
   "patient-free" AUC, and names any task the screen missed.
**C. What do the image models encode?** Without any diagnosis label: can the embeddings tell the scanner maker, the
   slice thickness, the field of view, the kVp and the kernel, even with the patient removed? Measured inside each
   collection too, so that "which collection" alone cannot explain it.

**Rehearsal first.** With the default `RUN_MODE = "both"`, experiment A first runs on **shuffled labels** (a rehearsal:
every number should look like chance), checks itself, and only then runs on the real labels. Experiment C uses no
diagnosis labels and runs once, at the end.

**How to run:** in Colab choose *Runtime → Change runtime type → L4 GPU (or A100), High-RAM*, then *Runtime → Run all*.
Sign in with **the Google account that holds the project Drive folder** when Drive asks. Expect about 4–5 hours: about 3–4 hours to download and
prepare 725 new scans (92 GB), then about 20 minutes per pass, then about 15 minutes for experiment C. **If Colab disconnects, run all
again:** finished scans and finished tasks are loaded, not redone. Outputs go to `results/07_screen_validation/`.

In [ ]:
# 1. Setup: Drive, folders, GPU
import os, re, sys, json, time, copy, shutil, zipfile, hashlib, datetime, platform, warnings, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
    WORK = Path('/content/asa07_work')          # Colab's own disk: DICOM and NIfTI files, never Drive
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    WORK = Path(os.environ['ASA_WORK_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
IN05 = PROJECT / 'results' / '05_image_check'
IN06 = PROJECT / 'results' / '06_image_posthoc'
CACHE05, CACHE06 = IN05 / 'cache', IN06 / 'cache'     # read only
OUT_ROOT = PROJECT / 'results' / '07_screen_validation'
CACHE = OUT_ROOT / 'cache'                       # label-free: headers, embeddings and QC pictures of the new scans
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for d in (IN05, IN06):
    for m in ('rehearsal', 'real'):
        assert (d / m / 'run_manifest.json').exists(), f'{d / m} is missing: run notebooks 05 and 06 first.'
for d in (CACHE / 'headers', CACHE / 'embeddings', CACHE / 'qc', WORK / 'nifti', WORK / 'dicom'):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
free_gb = shutil.disk_usage(WORK).free / 1e9
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Work disk     :', WORK, f'({free_gb:.0f} GB free)')
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')
if free_gb < 120:
    print('WARNING: less than 120 GB free on the work disk. The prepared scans need about 60 GB there at the peak. '
          'Use an L4 or A100 runtime (they have more disk).')

In [ ]:
# 2. Packages (exact versions; installed into Colab only) (notebook 05, unchanged)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1', 'SimpleITK==2.5.6',
                    'pydicom==3.0.2', 'nibabel==5.4.2', 'merlin-vlm==0.0.7'], check=True)

import numpy as np
import pandas as pd
import requests
import scipy
from scipy import ndimage, sparse
from scipy.stats import rankdata, spearmanr
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed
import pydicom
import SimpleITK as sitk
import nibabel
import torch
import torchvision
import monai
from monai.transforms import (Compose, LoadImage, EnsureType, Orientation, Spacing, ScaleIntensityRange, SpatialCrop)
from monai.transforms.utils import generate_spatial_bounding_box
from monai.inferers import SlidingWindowSplitter
from monai.networks.nets.segresnet_ds import SegResEncoder
import huggingface_hub
from huggingface_hub import hf_hub_download
import safetensors.torch
import merlin
from merlin.models import i3res
from merlin.data.monai_transforms import ImageTransforms as MERLIN_TRANSFORMS
import importlib.metadata

warnings.filterwarnings('ignore', message='.*pixdim.*')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version.VersionString(), 'pydicom': pydicom.__version__,
            'nibabel': nibabel.__version__, 'huggingface_hub': huggingface_hub.__version__,
            'merlin-vlm': importlib.metadata.version('merlin-vlm'), 'cpus': N_CPU,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)
if IN_COLAB and DEVICE != 'cuda':
    print('WARNING: no GPU. Choose Runtime → Change runtime type → L4 GPU, then run all again.')

## Settings (plan sections 7d, 7f and 7g)
Every model, image and encoder setting is notebook 05's (checked against its run record below); the silhouette is
notebook 06's. New here: the task list of experiment A (fixed by rule, checked below), the test across tasks, and the
targets of experiment C.

In [ ]:
# 3. Settings
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}

TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
PREP_JOBS = max(1, min(8, N_CPU - 1))
EMB_WORKERS = max(1, min(6, N_CPU - 2)) if IN_COLAB else 0
BODY_HU, OPEN_MM, DILATE_MM, AIR_HU = -500, 3.0, 5.0, -1000
N_QC = 2
CTFM_REPO, CTFM_REVISION = 'project-lighter/ct_fm_feature_extractor', '527b0d1c262b433396c85ff2de84c427b26945a1'
CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP, CTFM_BATCH = (3.0, 1.0, 1.0), (24, 128, 128), 0.625, 32
MERLIN_REPO, MERLIN_REVISION = 'stanfordmimi/Merlin', '66dd07585de79e4d8d70d67806da88f3ccaa5a0c'
MERLIN_CKPT = 'i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt'
ENCODERS = ('ctfm', 'merlin')
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL
GATE_NULL_RANGE = (0.35, 0.55)
GATE_MIN_COVERAGE = 0.95
REPRO_TOL = 1e-3            # as notebook 06 (plan 7f, settled while building, item 1)
SIL_HU = 0                  # notebook 06's silhouette

PREP_KEY05 = hashlib.md5(json.dumps({'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                                     'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY05 = hashlib.md5(json.dumps({'prep': PREP_KEY05, 'ctfm': [CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                                    'merlin': [MERLIN_REVISION, MERLIN_CKPT], 'v': 1},
                                   sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY = hashlib.md5(json.dumps({'nb05': EMB_KEY05, 'SIL_HU': SIL_HU, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
PREP_KEY = hashlib.md5(json.dumps({'nb05': PREP_KEY05, 'SIL_HU': SIL_HU, 'nb07': 1},
                                  sort_keys=True).encode()).hexdigest()[:12]
VARIANTS = ('original', 'background', 'silhouette')
INPUT_NAME = {'original': 'image', 'background': 'background', 'silhouette': 'silhouette'}

# Experiment A (plan 7g): every CT task of the notebook 03 audit that was not flagged. The list is derived by that rule
# below and must equal this one (written into the plan before any of these scans was downloaded).
EXPECTED_A = ['LIDC-IDRI | benign vs malignant', 'TCGA-LIHC | grade', 'TCGA-BLCA | stage', 'TCGA-KIRC | stage',
              'NSCLC Radiogenomics | histology', 'NSCLC Radiogenomics | EGFR mutation', 'TCGA-LIHC | stage',
              'TCGA-KIRC | grade', 'TCGA-UCEC | grade', 'HCC-TACE-Seg | stage', 'CPTAC-CCRCC | grade',
              'NSCLC Radiogenomics | KRAS mutation', 'CPTAC-CCRCC | stage', 'CPTAC-UCEC | grade', 'CPTAC-PDA | grade']
N_TASK_PERM = 99_999        # the test across tasks: random permutations of the task labels

# Experiment C (plan 7g; label-free): targets from the DICOM headers; values with fewer scans are left out.
DEC_MIN_LEVEL = 20

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'patients': [p for p in os.environ.get('ASA_TEST_PATIENTS', '').split(';') if p],
            'fake_images': os.environ.get('ASA_TEST_FAKE_IMAGES') == '1',
            'stop_after_images': os.environ.get('ASA_TEST_STOP_AFTER_IMAGES') == '1',
            'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    N_TASK_PERM = int(os.environ.get('ASA_TEST_N_TASK_PERM', N_TASK_PERM))
    CTFM_BATCH = int(os.environ.get('ASA_TEST_CTFM_BATCH', CTFM_BATCH))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT, '| N_TASK_PERM', N_TASK_PERM)

In [ ]:
# 4. Load and verify the inputs from notebooks 01-06; the tasks of experiment A
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

def cache_md5(folder):
    files = sorted((q for q in folder.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                   key=lambda q: q.relative_to(folder).parts)
    return len(files), hashlib.md5(''.join(md5(q) for q in files).encode()).hexdigest()

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
man05 = {m: verify(IN05 / m) for m in ('rehearsal', 'real')}
man06 = {m: verify(IN06 / m) for m in ('rehearsal', 'real')}
runs01_04 = {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
             '04': man04['run_started_utc']}
for m in ('rehearsal', 'real'):
    assert man05[m]['input_runs'] == runs01_04, f'notebook 05 ({m}) was run on other inputs'
    assert man06[m]['input_runs'] == {**runs01_04, '05': man05['real']['run_started_utc']}, \
        f'notebook 06 ({m}) was run on other inputs'
    if not TEST:
        for name, man in (('05', man05[m]), ('06', man06[m])):
            assert man['in_colab'] and not man['local_test_settings'], f'notebook {name} {m} is not the real Colab run'
assert man05['real']['labels'] == 'real' and man05['rehearsal']['labels'].startswith('shuffled')
s05 = man05['real']['settings']
same = {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'MIN_COUNT': MIN_COUNT,
        'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS, 'BODY_HU': BODY_HU,
        'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU, 'HDR_COLS': HDR_COLS, 'PREP_KEY': PREP_KEY05,
        'EMB_KEY': EMB_KEY05, 'CTFM': [CTFM_REPO, CTFM_REVISION, list(CTFM_SPACING), list(CTFM_PATCH), CTFM_OVERLAP],
        'MERLIN': [MERLIN_REPO, MERLIN_REVISION, MERLIN_CKPT]}
if not TEST:
    same.update({'N_BOOT': N_BOOT, 'N_PERM': N_PERM})
diff = [k for k, v in same.items() if json.loads(json.dumps(s05[k])) != json.loads(json.dumps(v))]
assert not diff, f'settings differ from notebook 05: {diff}'
s06 = man06['real']['settings']
assert (s06['SIL_HU'], s06['EMB_KEY06']) == (SIL_HU, EMB_KEY), 'silhouette settings differ from notebook 06'

n05, md5_05 = cache_md5(CACHE05)
assert (n05, md5_05) == (man05['real']['cache_files']['count'], man05['real']['cache_files']['combined_md5']), \
    'notebook 05 cache differs from its run record'
n06, md5_06 = cache_md5(CACHE06)
assert (n06, md5_06) == (man06['real']['cache_files']['count'], man06['real']['cache_files']['combined_md5']), \
    'notebook 06 cache differs from its run record'
v05 = man05['real']['versions']
same_versions = (v05['numpy'], v05['pandas'], v05['scipy'], v05['scikit-learn']) == \
                (np.__version__, pd.__version__, scipy.__version__, sklearn.__version__)
print(f'Inputs verified: notebooks 01-06, the notebook 05 cache ({n05} files) and the notebook 06 cache ({n06} files)')
print('Package versions', 'match notebook 05' if same_versions else
      f"DIFFER from notebook 05 (then numpy {v05['numpy']}, pandas {v05['pandas']}, scipy {v05['scipy']}, "
      f"scikit-learn {v05['scikit-learn']})")
if not man06['real'].get('reproduced_notebook05', False):
    print('NOTE: notebook 06 recorded that it did not reproduce notebook 05 within its margin (see its manifest).')

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
smeta = pd.read_csv(IN01 / 'series_metadata.csv.gz', usecols=['SeriesInstanceUID', 'FileSize']).drop_duplicates(
    'SeriesInstanceUID')
KEY = ['collection', 'PatientID', 'modality']
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)

# the tasks, by the rule of plan 7g
a03 = pd.read_csv(IN03 / 'audit_primary.csv')
modality_of = labels.groupby('task_id').modality.agg(lambda s: '/'.join(sorted(set(s))))
TASKS_A = [t for t in a03[~a03.flagged].task_id if modality_of[t] == 'CT']
FLAGGED = sorted(a03[a03.flagged].task_id)
assert sorted(TASKS_A) == sorted(EXPECTED_A), f'the unflagged CT tasks differ from plan 7g: {sorted(TASKS_A)}'
assert FLAGGED == sorted(man05['real']['tasks']), 'the flagged tasks differ from notebook 05'
meta_auc = a03.set_index('task_id').auc

lab = labels[labels.task_id.isin(TASKS_A)]
# several tasks share patients (one main series per patient): many task rows to one series
pts = lab.merge(main[KEY + ['SeriesInstanceUID', 'ImageCount']], on=KEY, how='left', validate='many_to_one')
pts = pts.merge(smeta, on='SeriesInstanceUID', how='left')
assert pts.SeriesInstanceUID.notna().all(), 'a task patient has no main series'
assert not pts.duplicated(['task_id', 'key'] if 'key' in pts else ['task_id'] + KEY).any(), 'a patient twice in a task'
pts['key'] = pts.collection + '__' + pts.PatientID
inv05 = pd.read_csv(CACHE05 / 'image_inventory.csv')
IN05_KEYS = set(inv05[inv05.embedded].key)
PATIENTS = pts.drop_duplicates('key').sort_values('key').reset_index(drop=True)
NEW = PATIENTS[~PATIENTS.key.isin(IN05_KEYS)].reset_index(drop=True)
if TEST.get('patients'):
    NEW = NEW[NEW.key.isin(TEST['patients'])].reset_index(drop=True)
QC_KEYS = set(pts[pts.key.isin(NEW.key)].sort_values('key').groupby('task_id').key.head(N_QC))
print(f'Experiment A: {len(TASKS_A)} tasks, {len(PATIENTS)} patients; {len(PATIENTS) - len(NEW)} already embedded in '
      f'notebooks 05-06; {len(NEW)} new ({NEW.FileSize.sum() / 1e9:.1f} GB of DICOM to download)')
show(pts.groupby('task_id').agg(patients=('key', 'size'), new=('key', lambda s: int((~s.isin(IN05_KEYS)).sum())),
                                metadata_auc=('task_id', lambda s: round(float(meta_auc[s.iloc[0]]), 3))))

## Part 1 · Download and prepare the new scans (no labels used)
Same series rule, reading, body mask, background-only version and silhouette as notebooks 05 and 06. Each new scan's
DICOM-header fields are saved like notebook 05's. The scans of notebook 05 are not downloaded again.

In [ ]:
# 5. Download, read and convert one series (notebook 05, unchanged)
def download_series(uid, dest):
    last = None
    for wait in (0, 10, 30, 90, 240):
        time.sleep(wait)
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(f'{TCIA}/getImage', params={'SeriesInstanceUID': uid}, stream=True,
                              timeout=(30, 600)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                badf = z.testzip()
                assert badf is None, f'corrupt file in zip: {badf}'
            tmp.replace(dest)
            return dest.stat().st_size
        except Exception as e:   # network errors, partial files: try again
            last = e
    raise RuntimeError(f'download failed 5 times: {last!r}')

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def _text(v):
    if v is None:
        return np.nan
    if isinstance(v, pydicom.multival.MultiValue):
        v = '/'.join(str(x) for x in v)
    v = str(v).strip().upper()
    return v if v else np.nan

def read_series(folder, uid):
    """The slices of one series, sorted along the slice normal, with geometry checks."""
    metas = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != uid:
            continue
        if not all(t in ds for t in ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing', 'Rows')):
            continue
        metas.append((p, ds))
    assert metas, 'no image slices in the download'
    combo = [(tuple(np.round(np.asarray(ds.ImageOrientationPatient, float), 3)), int(ds.Rows), int(ds.Columns))
             for _, ds in metas]
    major = pd.Series(combo).value_counts().index[0]
    keep = [m for m, c in zip(metas, combo) if c == major]
    iop = np.asarray(keep[0][1].ImageOrientationPatient, float)
    row_cos, col_cos = iop[:3], iop[3:]
    normal = np.cross(row_cos, col_cos)
    pos = np.array([[float(v) for v in ds.ImagePositionPatient] for _, ds in keep])
    proj = pos @ normal
    inst = np.array([_num(getattr(ds, 'InstanceNumber', np.nan)) for _, ds in keep])
    order = np.lexsort((np.nan_to_num(inst, nan=1e9), proj))
    dup = np.r_[False, np.diff(proj[order]) < 1e-3]          # same position twice: keep the first
    order = order[~dup]
    keep, pos, proj = [keep[i] for i in order], pos[order], proj[order]
    assert len(keep) >= 10, f'only {len(keep)} usable slices'
    gaps = np.diff(proj)
    z = float(np.median(gaps))
    step = pos[-1] - pos[0]
    geo = {'row_cos': row_cos, 'col_cos': col_cos, 'normal': normal, 'origin': pos[0], 'z': z,
           'n_other_orientation_dropped': sum(c != major for c in combo),
           'n_duplicates_dropped': int(dup.sum()),
           'max_gap_deviation_mm': float(np.max(np.abs(gaps - z))),
           'nonuniform_spacing': bool(np.max(np.abs(gaps - z)) > max(0.05, 0.1 * z)),
           'gantry_tilt_deg': float(np.degrees(np.arccos(min(1.0, abs(step @ normal) / np.linalg.norm(step))))),
           'coverage_mm': float(proj[-1] - proj[0] + z)}
    return keep, geo

def header_fields(keep, geo):
    ds0 = keep[0][1]
    def med(name):
        vals = [_num(getattr(ds, name, None)) for _, ds in keep]
        vals = [v for v in vals if np.isfinite(v)]
        return float(np.median(vals)) if vals else np.nan
    ps = [float(v) for v in ds0.PixelSpacing]
    agent = getattr(ds0, 'ContrastBolusAgent', None)
    return {'hdr_manufacturer': _text(getattr(ds0, 'Manufacturer', None)),
            'hdr_model': _text(getattr(ds0, 'ManufacturerModelName', None)),
            'hdr_slice_thickness': med('SliceThickness'), 'hdr_z_spacing': geo['z'],
            'hdr_pixel_spacing': float(np.mean(ps)), 'hdr_rows': int(ds0.Rows), 'hdr_columns': int(ds0.Columns),
            'hdr_n_slices': len(keep), 'hdr_coverage_mm': geo['coverage_mm'],
            'hdr_recon_diameter': med('ReconstructionDiameter'), 'hdr_kvp': med('KVP'),
            'hdr_tube_current': med('XRayTubeCurrent'), 'hdr_exposure': med('Exposure'),
            'hdr_ctdivol': med('CTDIvol'), 'hdr_pitch': med('SpiralPitchFactor'),
            'hdr_kernel': _text(getattr(ds0, 'ConvolutionKernel', None)),
            'hdr_filter_type': _text(getattr(ds0, 'FilterType', None)),
            'hdr_patient_position': _text(getattr(ds0, 'PatientPosition', None)),
            'hdr_contrast_agent_given': ('yes' if agent is not None and str(agent).strip() else 'no'),
            'hdr_gantry_tilt_deg': geo['gantry_tilt_deg'], 'hdr_nonuniform_spacing': geo['nonuniform_spacing'],
            'hdr_max_gap_deviation_mm': geo['max_gap_deviation_mm'],
            'hdr_duplicates_dropped': geo['n_duplicates_dropped'],
            'hdr_other_orientation_dropped': geo['n_other_orientation_dropped']}

def to_image(keep, geo):
    """HU volume with the geometry from the DICOM positions (not from the reader's guess)."""
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(p) for p, _ in keep])
    reader.SetOutputPixelType(sitk.sitkFloat32)          # GDCM applies the rescale slope and intercept (HU)
    img = reader.Execute()
    ps = [float(v) for v in keep[0][1].PixelSpacing]      # [between rows, between columns]
    img.SetSpacing((ps[1], ps[0], geo['z']))
    img.SetOrigin(tuple(float(v) for v in geo['origin']))
    img.SetDirection(tuple(np.stack([geo['row_cos'], geo['col_cos'], geo['normal']], axis=1).ravel()))
    return img

def like(arr, ref, dtype=np.int16):
    """A SimpleITK image from a (z, y, x) array, with the geometry of `ref`."""
    if dtype == np.int16:
        arr = np.clip(np.round(arr), -32768, 32767)
    out = sitk.GetImageFromArray(arr.astype(dtype))
    out.CopyInformation(ref)
    return out

In [ ]:
# 6. Body mask (notebook 05, unchanged), background-only version and silhouette (notebook 06), QC picture
def body_mask(arr, sp):
    """arr: (z, y, x) HU; sp: (sz, sy, sx) mm. Returns (body, body dilated by DILATE_MM)."""
    m = arr > BODY_HU
    r = max(1, int(round(OPEN_MM / ((sp[1] + sp[2]) / 2))))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    m = ndimage.binary_opening(m, structure=(yy ** 2 + xx ** 2 <= r * r)[None])
    lab_, n = ndimage.label(m)
    if n == 0:
        return np.zeros_like(m), np.zeros_like(m)
    counts = np.bincount(lab_.ravel())
    counts[0] = 0
    largest = lab_ == counts.argmax()
    del lab_, m
    core = ndimage.binary_fill_holes(largest, structure=np.ones((1, 3, 3), bool))
    del largest
    # dilation by DILATE_MM, computed inside the bounding box plus a margin (same result, much faster)
    pad = [int(np.ceil(DILATE_MM / s)) + 1 for s in sp]
    idx = np.where(core)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, core.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    dil = np.zeros_like(core)
    dil[box] = ndimage.distance_transform_edt(~core[box], sampling=sp).astype(np.float32) <= DILATE_MM
    return core, dil

def silhouette(dil):
    """Notebook 06: the region the background-only version blanked, as a flat shape; air outside."""
    return np.where(dil, SIL_HU, AIR_HU).astype(np.int16)

def qc_picture(key, arr, core, bg, sil, path):
    from matplotlib.figure import Figure
    z = int(np.argmax(core.sum(axis=(1, 2))))
    fig = Figure(figsize=(12, 4.2))
    ax = fig.subplots(1, 3)
    kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(core[z], levels=[0.5], colors='r', linewidths=0.6)
    ax[0].set_title('original + body outline')
    ax[1].imshow(bg[z], **kw); ax[1].set_title('background only')
    ax[2].imshow(sil[z], **kw); ax[2].set_title('silhouette')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=80)

def nifti_paths(key):
    d = WORK / 'nifti'
    return {v: d / f'{key}__{v}.nii.gz' for v in VARIANTS}

def header_path(key):
    return CACHE / 'headers' / f'{key}.json'

def header_ok(key):
    p = header_path(key)
    return p.exists() and json.loads(p.read_text()).get('prep_key') == PREP_KEY

In [ ]:
# 7. Prepare one new patient (runs in parallel workers; notebook 05's steps plus the silhouette)
def prepare_patient(key, uid, file_size):
    paths = nifti_paths(key)
    if header_ok(key) and all(p.exists() for p in paths.values()):
        return {'key': key, 'status': 'ready'}
    t0 = time.time()
    tmp = WORK / 'dicom' / key
    zpath = WORK / 'dicom' / f'{key}.zip'
    try:
        shutil.rmtree(tmp, ignore_errors=True)
        nbytes = download_series(uid, zpath)
        with zipfile.ZipFile(zpath) as z:
            n_files = len([n for n in z.namelist() if not n.endswith('/')])
            z.extractall(tmp)
        keep, geo = read_series(tmp, uid)
        hdr = header_fields(keep, geo)
        img = to_image(keep, geo)
        arr = sitk.GetArrayFromImage(img)
        sp = img.GetSpacing()[::-1]
        core, dil = body_mask(arr, sp)
        bg = arr.copy()
        bg[dil] = AIR_HU
        sil = silhouette(dil)
        for name, a in (('original', arr), ('background', bg), ('silhouette', sil)):
            sitk.WriteImage(like(a, img, np.int16), str(paths[name]), useCompression=True)
        area = core.sum(axis=(1, 2)) * sp[1] * sp[2] / 100.0
        sub = arr[::2, ::4, ::4]
        rec = {'key': key, 'SeriesInstanceUID': uid, 'prep_key': PREP_KEY, 'download_bytes': nbytes,
               'tcia_file_size': file_size, 'n_files_in_zip': n_files, **hdr,
               'body_area_cm2': float(area[area > 0].mean()) if (area > 0).any() else np.nan,
               'body_fraction_dilated': float(dil.mean()),
               'hu_p01': float(np.percentile(sub, 1)), 'hu_p50': float(np.percentile(sub, 50)),
               'hu_p99': float(np.percentile(sub, 99)), 'prep_seconds': round(time.time() - t0, 1)}
        if key in QC_KEYS:
            qc_picture(key, arr, core, bg, sil, CACHE / 'qc' / f'{key}.png')
        write_json_atomic(header_path(key), rec)
        return {'key': key, 'status': 'ok', 'seconds': rec['prep_seconds']}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        zpath.unlink(missing_ok=True)

### Encoders (notebook 05, unchanged)

In [ ]:
# 8. Load the two encoders (notebook 05, unchanged)
i3res.checkpoint = type('NoCheckpoint', (), {'checkpoint': staticmethod(lambda fn, *a, **k: fn(*a))})

def load_ctfm():
    cfg = json.loads(Path(hf_hub_download(CTFM_REPO, 'config.json', revision=CTFM_REVISION)).read_text())
    model = SegResEncoder(**cfg)
    state = safetensors.torch.load_file(hf_hub_download(CTFM_REPO, 'model.safetensors', revision=CTFM_REVISION))
    model.load_state_dict(state, strict=True)
    return model.eval()

MERLIN_DIR = Path(merlin.__file__).parent / 'models' / 'checkpoints'   # where the official class looks for it
if not IN_COLAB and os.environ.get('ASA_TEST_MERLIN_DIR'):              # local tests: Windows path-length limit
    MERLIN_DIR = Path(os.environ['ASA_TEST_MERLIN_DIR'])

def load_merlin():
    ckpt = hf_hub_download(MERLIN_REPO, MERLIN_CKPT, revision=MERLIN_REVISION, local_dir=MERLIN_DIR)
    state = torch.load(ckpt, map_location='cpu')
    prefix = 'encode_image.i3_resnet.'
    enc = i3res.I3ResNet(copy.deepcopy(torchvision.models.resnet152(weights=None)), class_nb=1692, conv_class=True,
                         ImageEmbedding=True)
    enc.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}, strict=True)
    return enc.eval()

# CT-FM preprocessing (authors' script), split so that the background-only version can use the original's crop box
CTFM_LOAD = Compose([LoadImage(ensure_channel_first=True), EnsureType(), Orientation(axcodes='SPL'),
                     Spacing(pixdim=CTFM_SPACING, mode='bilinear')])
CTFM_SCALE = ScaleIntensityRange(a_min=-1024, a_max=2048, b_min=0, b_max=1, clip=True)

def ctfm_inputs(paths, crop_from=None):
    """Preprocessed CT-FM inputs for each variant. With crop_from, every variant uses that variant's foreground box."""
    imgs = {v: CTFM_LOAD(str(p)) for v, p in paths.items()}
    out, boxes = {}, {}
    for v, img in imgs.items():
        src = imgs[crop_from] if crop_from else img
        start, end = generate_spatial_bounding_box(src, select_fn=lambda x: x > 0)   # CropForeground's default
        out[v] = _plain(CTFM_SCALE(SpatialCrop(roi_start=start, roi_end=end)(img)))
        boxes[v] = [int(a) for a in start] + [int(b) for b in end]
    return out, boxes

def _plain(t):
    return (t.as_tensor() if hasattr(t, 'as_tensor') else torch.as_tensor(t)).float()

def merlin_inputs(paths):
    return {v: _plain(MERLIN_TRANSFORMS({'image': str(p)})['image']) for v, p in paths.items()}

class PrepDataset(torch.utils.data.Dataset):
    """CPU preprocessing in DataLoader workers; the GPU only runs the encoders."""
    def __init__(self, items):
        self.items = items              # (key, {variant: path}, crop_from)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        key, paths, crop_from = self.items[i]
        try:
            ct, boxes = ctfm_inputs(paths, crop_from)
            return {'key': key, 'ctfm': ct, 'merlin': merlin_inputs(paths), 'boxes': boxes, 'error': None}
        except Exception as e:
            root = e
            while root.__cause__ is not None:      # MONAI wraps the real error; report the root cause
                root = root.__cause__
            return {'key': key, 'error': f'{e!r} | root cause: {root!r}'[:600]}

@torch.no_grad()
def ctfm_embed(model, vol):
    feats, batch = [], []
    def run(b):
        x = torch.cat(b).to(DEVICE)
        return torch.nn.functional.adaptive_avg_pool3d(model(x)[-1], 1).flatten(1).cpu()
    for patch, _ in SlidingWindowSplitter(CTFM_PATCH, CTFM_OVERLAP)(vol.unsqueeze(0)):
        batch.append(patch)
        if len(batch) == CTFM_BATCH:
            feats.append(run(batch)); batch = []
    if batch:
        feats.append(run(batch))
    f = torch.cat(feats)
    return f.mean(0).numpy().astype(np.float32), int(len(f))

@torch.no_grad()
def merlin_embed(model, vol):
    return model(vol.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy().astype(np.float32)

def embed_items(items, out_path_of, models, extra_of=None, label='scans'):
    """Embed every item with both encoders and save one .npz per item (resume point)."""
    loader = torch.utils.data.DataLoader(PrepDataset(items), batch_size=None, shuffle=False,
                                         num_workers=EMB_WORKERS if len(items) > 1 else 0)
    done, failed, t0 = 0, [], time.time()
    for rec in loader:
        if rec['error']:
            failed.append({'key': rec['key'], 'error': rec['error']})
            continue
        arrays, meta = {}, {'emb_key': EMB_KEY, 'boxes': rec['boxes'], **(extra_of(rec['key']) if extra_of else {})}
        for v, vol in rec['ctfm'].items():
            arrays[f'ctfm__{v}'], meta[f'ctfm__{v}__patches'] = ctfm_embed(models['ctfm'], vol)
        for v, vol in rec['merlin'].items():
            arrays[f'merlin__{v}'] = merlin_embed(models['merlin'], vol)
        path = out_path_of(rec['key'])
        tmp = path.with_name(path.stem + '.tmp.npz')
        np.savez(tmp, meta=json.dumps(meta, default=_jsonable), **arrays)
        tmp.replace(path)
        done += 1
        if done % 25 == 0 or done == len(items):
            print(f'  {label}: {done}/{len(items)} embedded ({(time.time() - t0) / 60:.1f} min)')
    return failed

def emb_path(key):
    return CACHE / 'embeddings' / f'{key}.npz'

def emb_ok(path, variants):
    if not path.exists():
        return False
    with np.load(path) as z:
        meta = json.loads(str(z['meta']))
        return meta.get('emb_key') == EMB_KEY and all(f'{e}__{v}' in z.files for e in ENCODERS for v in variants)

def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background), notebook 06 (silhouette), or this notebook."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    return emb_path(key)

def has_variant(key, variant):
    p = emb_file(key, variant)
    if not p.exists():
        return False
    with np.load(p) as z:
        ok_key = json.loads(str(z['meta'])).get('emb_key') in (EMB_KEY05, EMB_KEY)
        return ok_key and all(f'{e}__{variant}' in z.files for e in ENCODERS)

def header_of(key):
    p = (CACHE05 / 'headers' if key in IN05_KEYS else CACHE / 'headers') / f'{key}.json'
    return json.loads(p.read_text()) if p.exists() else {'key': key}

In [ ]:
# 9. Part 1: prepare and embed the new scans (label-free; resumable)
def fake_part1():
    """Local code tests only: random headers and embeddings instead of images (seeded, label-free)."""
    rng = np.random.default_rng(7)
    for key in NEW.key:
        if not header_ok(key):
            write_json_atomic(header_path(key), {
                'key': key, 'prep_key': PREP_KEY, 'fake': True,
                'hdr_manufacturer': str(rng.choice(['GE MEDICAL SYSTEMS', 'SIEMENS', 'PHILIPS', 'TOSHIBA'])),
                'hdr_z_spacing': float(rng.choice([0.625, 1.25, 2.5, 5.0])), 'hdr_kvp': float(rng.choice([100, 120, 140])),
                'hdr_slice_thickness': float(rng.choice([1.25, 2.5, 5.0])), 'hdr_pixel_spacing': float(rng.uniform(.6, .9)),
                'hdr_recon_diameter': float(rng.uniform(300, 450)), 'hdr_tube_current': float(rng.uniform(100, 400)),
                'hdr_exposure': np.nan, 'hdr_ctdivol': np.nan, 'hdr_pitch': float(rng.choice([0.984, 1.375])),
                'hdr_kernel': str(rng.choice(['B30F', 'STANDARD', 'B', 'LUNG'])), 'hdr_filter_type': np.nan,
                'hdr_patient_position': 'FFS', 'hdr_contrast_agent_given': str(rng.choice(['yes', 'no'])),
                'body_area_cm2': float(rng.uniform(400, 900))})
        if not emb_ok(emb_path(key), VARIANTS):
            arrays = {f'ctfm__{v}': rng.normal(size=512).astype(np.float32) for v in VARIANTS}
            arrays.update({f'merlin__{v}': np.abs(rng.normal(size=2048)).astype(np.float32) for v in VARIANTS})
            np.savez(emb_path(key), meta=json.dumps({'emb_key': EMB_KEY, 'fake': True}), **arrays)

k05 = next(iter(sorted(IN05_KEYS)))
with np.load(CACHE05 / 'embeddings' / f'{k05}.npz') as z:
    assert json.loads(str(z['meta']))['emb_key'] == EMB_KEY05, 'notebook 05 embeddings were made with other settings'
with np.load(CACHE06 / 'embeddings' / f'{k05}.npz') as z:
    assert json.loads(str(z['meta']))['emb_key'] == EMB_KEY, 'notebook 06 embeddings were made with other settings'

t_part1 = time.time()
prep_failed, emb_failed = [], []
if TEST.get('fake_images'):
    fake_part1()
    MODELS = None
else:
    todo = [r for r in NEW.itertuples() if not emb_ok(emb_path(r.key), VARIANTS) or not header_ok(r.key)]
    print(f'New scans to prepare: {len(todo)} of {len(NEW)} (the others are finished in the cache)')
    if todo:
        res = Parallel(n_jobs=PREP_JOBS, return_as='generator_unordered')(
            delayed(prepare_patient)(r.key, r.SeriesInstanceUID, r.FileSize) for r in todo)
        for i, r in enumerate(res, 1):
            if r['status'] == 'failed':
                prep_failed.append(r)
                print(f"  FAILED {r['key']}: {r['error']}")
            if i % 25 == 0 or i == len(todo):
                print(f'  prepared {i}/{len(todo)} ({(time.time() - t_part1) / 60:.1f} min, '
                      f'{shutil.disk_usage(WORK).free / 1e9:.0f} GB free)')
    MODELS = {'ctfm': load_ctfm().to(DEVICE), 'merlin': load_merlin().to(DEVICE)}
    todo = [k for k in NEW.key if not emb_ok(emb_path(k), VARIANTS)
            and all(p.exists() for p in nifti_paths(k).values())]
    print(f'New scans to embed: {len(todo)}')
    items = [(k, nifti_paths(k), 'original') for k in todo]   # background and silhouette use the original's crop box
    emb_failed = embed_items(items, emb_path, MODELS, label='original + background + silhouette')
    for f in emb_failed:
        print(f"  EMBEDDING FAILED {f['key']}: {f['error']}")
    for k in NEW.key:                    # embedded scans are not needed again: free the work disk
        if emb_ok(emb_path(k), VARIANTS):
            for p in nifti_paths(k).values():
                p.unlink(missing_ok=True)
print(f'Part 1 finished in {(time.time() - t_part1) / 60:.1f} min')

complete = {k for k in PATIENTS.key if all(has_variant(k, v) for v in VARIANTS)}
DEC_KEYS = sorted(k for k in IN05_KEYS | set(PATIENTS.key) if all(has_variant(k, v) for v in VARIANTS))
inventory = NEW[['key', 'collection', 'PatientID', 'SeriesInstanceUID', 'ImageCount', 'FileSize']].merge(
    pd.DataFrame([header_of(k) for k in NEW.key]).drop(columns=['SeriesInstanceUID'], errors='ignore'),
    on='key', how='left')
inventory['embedded'] = inventory.key.isin(complete)
inventory.to_csv(CACHE / 'image_inventory.csv', index=False)
pd.DataFrame(prep_failed + emb_failed, columns=['key', 'status', 'error']).to_csv(CACHE / 'failures_this_session.csv',
                                                                                  index=False)
cov = pts.assign(embedded=pts.key.isin(complete)).groupby('task_id').embedded.agg(['sum', 'size', 'mean'])
show(cov.rename(columns={'sum': 'embedded', 'size': 'patients', 'mean': 'coverage'}))
print(f'Scans with all three inputs, for experiment C: {len(DEC_KEYS)}')
if (cov['mean'] < GATE_MIN_COVERAGE).any():
    print(f'WARNING: fewer than {GATE_MIN_COVERAGE:.0%} of the patients of a task have embeddings. '
          'Run all again (failed downloads are retried); if it persists, inspect the failed-download log.')
if TEST.get('stop_after_images'):
    raise SystemExit('Local test: stopping after the image part.')

## Part 2 · Experiment A (once on shuffled labels as a rehearsal, then on the real labels)

In [ ]:
# 10. Model building blocks (identical to notebooks 03 and 04), plus the same models on embeddings (notebook 05, unchanged)
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [ ]:
# 11. Notebook 05's task frame, embedding loader and flag rule (unchanged), and the test across tasks
def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def flag(lo, q):
    return bool(lo >= FLAG_LOWER_CI and q < FLAG_Q)

def load_inputs(keys):
    """{(encoder, variant): matrix} from whichever notebook holds each scan's embedding."""
    out = {}
    for v in VARIANTS:
        out.update(load_embeddings(keys, (v,), lambda k, v=v: emb_file(k, v)))
    return out

def spearman_perm(a, b):
    """Spearman rho of two task-level vectors, with a one-sided test by random permutations of the tasks."""
    ok = ~(np.isnan(a) | np.isnan(b))
    ra, rb = rankdata(np.asarray(a)[ok]), rankdata(np.asarray(b)[ok])
    ra = ra - ra.mean()
    obs = float(ra @ (rb - rb.mean()) / np.sqrt((ra ** 2).sum() * ((rb - rb.mean()) ** 2).sum()))
    rng = np.random.default_rng(SEED)
    null = np.empty(N_TASK_PERM)
    for i0 in range(0, N_TASK_PERM, 10_000):
        P = np.array([rng.permutation(rb) for _ in range(min(10_000, N_TASK_PERM - i0))])
        P = P - P.mean(1, keepdims=True)
        null[i0:i0 + len(P)] = (P @ ra) / np.sqrt((P ** 2).sum(1) * (ra ** 2).sum())
    return obs, (1 + int((null >= obs - 1e-12).sum())) / (1 + N_TASK_PERM), int(ok.sum())

In [ ]:
# 12. Experiment A: every analysis for one task (both encoders, three inputs)
def analyse_task(tid, mode, shuffle_seed):
    df, y, classes = task_frame(tid, shuffle_seed)
    assert len(classes) == 2, f'{tid}: experiment A expects two classes'
    hdr_t = pd.DataFrame([header_of(k) for k in df.key])
    df = df.merge(hdr_t[['key', 'body_area_cm2']] if 'body_area_cm2' in hdr_t else hdr_t[['key']], on='key', how='left')
    if 'body_area_cm2' not in df:
        df['body_area_cm2'] = np.nan
    has = df.key.isin(complete).to_numpy()
    n_all = len(df)
    df, y = df[has].reset_index(drop=True), y[has]
    splits = repeated_splits(y)
    base = {'task_id': tid, 'n_patients': len(y), 'n_excluded_no_image': int(n_all - len(y)),
            'classes': ' | '.join(f'{c}: {int((y == j).sum())}' for j, c in enumerate(classes))}
    emb = load_inputs(df.key)
    P_meta = cv_oof_df(df, y, SCANNER_COLS, splits)
    P_body = cv_oof_df(df, y, ['body_area_cm2'], splits)
    lo_b, hi_b = boot_ci(y, P_body)
    m_auc = mean_auc(y, P_meta)
    d_meta = m_auc - float(meta_auc[tid]) if shuffle_seed is None else np.nan
    rec = {'probe': [], 'oof': []}
    for enc in ENCODERS:
        row = {**base, 'encoder': enc, 'metadata_auc_nb03': float(meta_auc[tid]),
               'metadata_auc_same_patients': m_auc, 'metadata_auc_minus_nb03': d_meta,
               'reproduced_nb03': (bool(n_all == len(y) and abs(d_meta) < REPRO_TOL) if shuffle_seed is None else None),
               'body_size_auc': mean_auc(y, P_body), 'body_size_ci_low': lo_b, 'body_size_ci_high': hi_b}
        oof = {'task_id': tid, 'encoder': enc, 'key': df.key, 'label': y, 'p_metadata': P_meta[:, :, 1].mean(0)}
        for v in VARIANTS:
            name, X = INPUT_NAME[v], emb[(enc, v)]
            P, _ = cv_oof_matrix(X, y, splits)
            lo, hi = boot_ci(y, P)
            obs, p, null = permutation_test(scaled_folds(X), y)
            row.update({f'{name}_auc': mean_auc(y, P), f'{name}_ci_low': lo, f'{name}_ci_high': hi,
                        f'{name}_perm_p': p, f'{name}_null_mean_auc': float(null.mean())})
            oof[f'p_{name}'] = P[:, :, 1].mean(0)
        rec['probe'].append(row)
        rec['oof'].append(pd.DataFrame(oof))
    return rec

In [ ]:
# 13. Experiment A: the screen validation over all 19 CT tasks
def reading(rho, p, n_missed):
    """Plan 7g, experiment A, fixed before the run."""
    if rho > 0 and p < 0.05:
        return 'the screen is supported' if n_missed == 0 else 'supported, with misses'
    return 'not supported'

def screen_validation(mode, probe):
    p05 = pd.read_csv(IN05 / mode / 'image_probe.csv').set_index(['task_id', 'encoder'])
    s06 = pd.read_csv(IN06 / mode / 'silhouette_probe.csv').set_index(['task_id', 'encoder'])
    rows = []
    for enc in ENCODERS:
        for tid in FLAGGED:
            r5, r6 = p05.loc[(tid, enc)], s06.loc[(tid, enc)]
            rows.append({'task_id': tid, 'encoder': enc, 'metadata_flagged': True, 'metadata_auc': float(meta_auc[tid]),
                         'n_patients': int(r5.n_patients), 'image_auc': r5.image_auc, 'image_flagged': bool(r5.image_flagged),
                         'background_auc': r5.background_auc, 'background_flagged': bool(r5.background_flagged),
                         'silhouette_auc': r6.silhouette_auc, 'silhouette_flagged': bool(r6.silhouette_flagged),
                         'source': 'notebooks 05 (image, background) and 06 (silhouette)'})
        for r in probe[probe.encoder == enc].itertuples():
            rows.append({'task_id': r.task_id, 'encoder': enc, 'metadata_flagged': False,
                         'metadata_auc': float(meta_auc[r.task_id]), 'n_patients': r.n_patients,
                         **{f'{n}_auc': getattr(r, f'{n}_auc') for n in INPUT_NAME.values()},
                         **{f'{n}_flagged': bool(getattr(r, f'{n}_flagged')) for n in INPUT_NAME.values()},
                         'source': 'notebook 07'})
    tasks = pd.DataFrame(rows)
    summary = []
    for enc in ENCODERS:
        t = tasks[tasks.encoder == enc]
        for name in ('background', 'image', 'silhouette'):
            rho, p, n = spearman_perm(t.metadata_auc.to_numpy(float), t[f'{name}_auc'].to_numpy(float))
            mf, xf = t.metadata_flagged.to_numpy(), t[f'{name}_flagged'].to_numpy()
            missed = t.task_id[~mf & xf].tolist()
            summary.append({'encoder': enc, 'input': name, 'role': {'background': 'primary' if enc == 'ctfm' else
                                                                    'secondary', 'image': 'descriptive',
                                                                    'silhouette': 'control'}[name],
                            'n_tasks': n, 'spearman_rho': rho, 'perm_p_one_sided': p,
                            'both_flagged': int((mf & xf).sum()), 'metadata_only_flagged': int((mf & ~xf).sum()),
                            'image_test_only_flagged': int((~mf & xf).sum()), 'neither': int((~mf & ~xf).sum()),
                            'missed_by_metadata_screen': ' | '.join(missed) or 'none',
                            'reading': reading(rho, p, len(missed)) if name == 'background' else ''})
    return tasks, pd.DataFrame(summary)

In [ ]:
# 14. Run experiment A in one mode (rehearsal on shuffled labels, or real), with BH, flags and saving
def run_mode(mode):
    out = OUT_ROOT / mode
    (out / 'per_task').mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    key = hashlib.md5(json.dumps({'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT,
                                  'N_PERM': N_PERM, 'LR': LR_PARAMS, 'EMB_KEY': EMB_KEY, 'EMB_KEY05': EMB_KEY05,
                                  'TEST': TEST, 'shuffle': shuffle_seed, 'complete': sorted(complete)},
                                 sort_keys=True, default=str).encode()).hexdigest()
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    allrows, oof = [], []
    t_all = time.time()
    for i, tid in enumerate(TASKS_A, 1):
        t0 = time.time()
        stem = re.sub(r'\W+', '_', tid).strip('_')
        saved = out / 'per_task' / f'{stem}.json'
        if saved.exists() and json.loads(saved.read_text()).get('settings_key') == key:
            rec = json.loads(saved.read_text())['rec']
            oof.append(pd.read_csv(out / 'per_task' / f'{stem}_oof.csv.gz'))
            print(f'{i:>2}/{len(TASKS_A)} {tid:<38} loaded from an earlier run')
        else:
            rec = analyse_task(tid, mode, shuffle_seed)
            o = pd.concat(rec.pop('oof'), ignore_index=True)
            o.to_csv(out / 'per_task' / f'{stem}_oof.csv.gz', index=False)
            oof.append(o)
            write_json_atomic(saved, {'settings_key': key, 'rec': rec})
            r = {x['encoder']: x for x in rec['probe']}
            print(f"{i:>2}/{len(TASKS_A)} {tid:<38} CT-FM {r['ctfm']['image_auc']:.3f} / bg {r['ctfm']['background_auc']:.3f}"
                  f" | Merlin {r['merlin']['image_auc']:.3f} / bg {r['merlin']['background_auc']:.3f}"
                  f"  ({(time.time() - t0) / 60:.1f} min)")
        allrows += rec['probe']
    print(f'Experiment A finished in {(time.time() - t_all) / 60:.1f} min')

    probe = pd.DataFrame(allrows)
    for name in INPUT_NAME.values():
        probe[f'{name}_perm_q_bh'] = np.nan
        for enc, idx in probe.groupby('encoder').groups.items():
            probe.loc[idx, f'{name}_perm_q_bh'] = bh(probe.loc[idx, f'{name}_perm_p'])
        probe[f'{name}_flagged'] = [flag(lo, q) for lo, q in zip(probe[f'{name}_ci_low'], probe[f'{name}_perm_q_bh'])]
    tasks, summary = screen_validation(mode, probe)
    tables = {'image_probe_unflagged_tasks.csv': probe, 'screen_validation_tasks.csv': tasks,
              'screen_validation_summary.csv': summary}
    for name, t in tables.items():
        t.to_csv(out / name, index=False)
    pd.concat(oof, ignore_index=True).to_csv(out / 'oof_image_predictions.csv.gz', index=False)
    show(probe[['task_id', 'encoder', 'n_patients', 'image_auc', 'image_flagged', 'background_auc', 'background_ci_low',
                'background_perm_q_bh', 'background_flagged', 'silhouette_auc', 'silhouette_flagged']], 30)
    show(summary, 10)

    outputs = []
    for p in sorted(q for q in out.rglob('*') if q.is_file()):
        if p.name == 'run_manifest.json':
            continue
        n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
        outputs.append({'file': p.relative_to(out).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
    n07, md5_07 = cache_md5(CACHE)
    manifest = {
        'settings_key': key, 'notebook': '07_screen_validation.ipynb', 'mode': mode, 'plan': '7g (experiment A)',
        'labels': ('real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal'
                   else 'shuffled (local code test)'), 'run_started_utc': RUN_STARTED,
        'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'in_colab': IN_COLAB, 'local_test_settings': TEST,
        'input_runs': {**runs01_04, '05': man05['real']['run_started_utc'], '06': man06['real']['run_started_utc']},
        'input_cache_md5': {'05': md5_05, '06': md5_06},
        'tasks_A': TASKS_A, 'flagged_tasks': FLAGGED, 'same_versions_as_notebook05': same_versions,
        'reproduced_notebook03': (bool(probe.reproduced_nb03.all()) if shuffle_seed is None else None),
        'patients_total': int(len(PATIENTS)), 'patients_new': int(len(NEW)), 'patients_embedded': int(len(complete)),
        'settings': {**same, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM, 'N_TASK_PERM': N_TASK_PERM, 'SIL_HU': SIL_HU,
                     'REPRO_TOL': REPRO_TOL, 'PREP_KEY07': PREP_KEY, 'EMB_KEY07': EMB_KEY},
        'versions': VERSIONS, 'cache_files': {'count': n07, 'combined_md5': md5_07}, 'outputs': outputs,
    }
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return probe, summary

In [ ]:
# 15. Experiment C (label-free): what the embeddings encode about the scanner
def within_collection_auc(y, P, coll):
    """Mean over classes (one vs rest) of the AUC over pairs from the same collection; mean over repeats."""
    k = P.shape[2]
    vals = []
    for r in range(P.shape[0]):
        per = [within_site_auc((y == j).astype(int), P[r][:, j], coll)[0] for j in range(k)]
        vals.append(np.nanmean(per) if np.isfinite(per).any() else np.nan)
    return float(np.nanmean(vals))

def decoding_targets(h):
    """Plan 7g, experiment C. Returns {name: (values, kind)}; missing values (None) mark scans left out."""
    def cut(s, rules):
        out = pd.Series(None, index=h.index, dtype=object)
        for mask, value in rules:
            out[mask.fillna(False).to_numpy(bool)] = value
        return out
    z, rd, kv = (pd.to_numeric(h[c], errors='coerce') for c in ('hdr_z_spacing', 'hdr_recon_diameter', 'hdr_kvp'))
    text = lambda v: str(v).strip() if pd.notna(v) and str(v).strip() else None
    t = {'manufacturer': h.hdr_manufacturer.map(lambda v: text(v).split()[0] if text(v) else None),
         'slice spacing (<2 / 2-<4 / >=4 mm)': cut(z, [(z < 2, '<2 mm'), ((z >= 2) & (z < 4), '2-<4 mm'),
                                                       (z >= 4, '>=4 mm')]),
         'reconstruction diameter (median split)': cut(rd, [(rd > rd.median(), 'above median'),
                                                            (rd <= rd.median(), 'at or below median')]),
         'kVp (120 vs other)': cut(kv, [(kv == 120, '120'), (kv.notna() & (kv != 120), 'other')]),
         'kernel': h.hdr_kernel.map(text),
         'contrast agent given (clinical, reported separately)': h.hdr_contrast_agent_given.map(
             lambda v: v if v in ('yes', 'no') else None)}
    return {name: (vals.astype(object), 'clinical' if name.startswith('contrast') else 'acquisition')
            for name, vals in t.items()}

def run_decoding():
    out = OUT_ROOT / 'decoding'
    out.mkdir(parents=True, exist_ok=True)
    key = hashlib.md5(json.dumps({'keys': DEC_KEYS, 'EMB_KEY': EMB_KEY, 'N_BOOT': N_BOOT, 'MIN': DEC_MIN_LEVEL,
                                  'TEST': TEST}, sort_keys=True, default=str).encode()).hexdigest()
    saved = out / 'acquisition_decoding.csv'
    man_p = out / 'run_manifest.json'
    if saved.exists() and man_p.exists() and json.loads(man_p.read_text()).get('settings_key') == key:
        print('Experiment C: loaded from an earlier run')
        return pd.read_csv(saved)
    t0 = time.time()
    h = pd.DataFrame([header_of(k) for k in DEC_KEYS])
    for c in ('hdr_manufacturer', 'hdr_kernel', 'hdr_contrast_agent_given', 'hdr_z_spacing', 'hdr_recon_diameter',
              'hdr_kvp'):
        if c not in h:
            h[c] = np.nan
    coll = np.array([k.split('__')[0] for k in DEC_KEYS], dtype=object)
    emb = load_inputs(DEC_KEYS)
    rows = []
    for name, (vals, kind) in decoding_targets(h).items():
        counts = vals.value_counts()
        levels = sorted(counts[counts >= DEC_MIN_LEVEL].index)
        keep = vals.isin(levels).to_numpy()
        if len(levels) < 2:
            rows.append({'target': name, 'kind': kind, 'note': 'fewer than two values with enough scans'})
            continue
        y = vals[keep].map({l: i for i, l in enumerate(levels)}).to_numpy()
        splits = repeated_splits(y)
        y_shuf = np.random.default_rng(999).permutation(y)
        base = {'target': name, 'kind': kind, 'n_scans': int(keep.sum()), 'n_collections': len(set(coll[keep])),
                'values': ' | '.join(f'{l}: {int(counts[l])}' for l in levels)}
        for enc in ENCODERS:
            for v in VARIANTS:
                X = emb[(enc, v)][keep]
                P, _ = cv_oof_matrix(X, y, splits)
                lo, hi = boot_ci(y, P)
                Ps, _ = cv_oof_matrix(X, y_shuf, repeated_splits(y_shuf))
                rows.append({**base, 'encoder': enc, 'input': INPUT_NAME[v], 'pooled_auc': mean_auc(y, P),
                             'pooled_ci_low': lo, 'pooled_ci_high': hi,
                             'within_collection_auc': within_collection_auc(y, P, coll[keep]),
                             'shuffled_target_auc': mean_auc(y_shuf, Ps)})
        print(f'  {name:<52} {int(keep.sum())} scans, {len(levels)} values ({(time.time() - t0) / 60:.1f} min)')
    res = pd.DataFrame(rows)
    res.to_csv(saved, index=False)
    man = {'settings_key': key, 'notebook': '07_screen_validation.ipynb', 'part': 'experiment C (label-free)',
           'plan': '7g (experiment C)', 'run_started_utc': RUN_STARTED,
           'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
           'in_colab': IN_COLAB, 'local_test_settings': TEST, 'n_scans': len(DEC_KEYS), 'DEC_MIN_LEVEL': DEC_MIN_LEVEL,
           'versions': VERSIONS,
           'outputs': [{'file': saved.name, 'bytes': saved.stat().st_size, 'rows': len(res), 'md5': md5(saved)}]}
    man_p.write_text(json.dumps(man, indent=2, default=str))
    print(f'Experiment C finished in {(time.time() - t0) / 60:.1f} min')
    return res

In [ ]:
# 16. Run: experiment A rehearsal (shuffled labels) -> automatic gate -> real labels; then experiment C
modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results:
        probe_r, _ = results['rehearsal']
        nulls = pd.concat([probe_r[f'{n}_null_mean_auc'] for n in INPUT_NAME.values()])
        problems = []
        if len(probe_r) != 2 * len(TASKS_A):
            problems.append('not every task and encoder finished')
        if not nulls.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {nulls.round(3).tolist()}')
        if (cov['mean'] < GATE_MIN_COVERAGE).any():
            problems.append('too few scans embedded for a task')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            print('Inspect the run log.')
            break
        print('\nRehearsal gate passed: all analyses ran, null means look like chance, enough scans.')
    results[mode] = run_mode(mode)
if not modes or 'real' not in modes or 'real' in results:
    decoding = run_decoding()
    show(decoding[[c for c in ('target', 'n_scans', 'encoder', 'input', 'pooled_auc', 'within_collection_auc',
                               'shuffled_target_auc') if c in decoding]], 40)

In [ ]:
# 17. Plain-English summary, read back from the saved files
for mode in results:
    out = OUT_ROOT / mode
    pr = pd.read_csv(out / 'image_probe_unflagged_tasks.csv')
    sm = pd.read_csv(out / 'screen_validation_summary.csv')
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind}{': every number should look like chance' if lab_kind.startswith('shuffled') else ''})")
    for enc, g in pr.groupby('encoder'):
        print(f"- {enc}, the 15 unflagged tasks: full image flagged in {int(g.image_flagged.sum())}, background only in "
              f"{int(g.background_flagged.sum())}, silhouette in {int(g.silhouette_flagged.sum())}; "
              f"null means {g.background_null_mean_auc.min():.3f}-{g.background_null_mean_auc.max():.3f}")
    for r in sm[sm.input == 'background'].itertuples():
        print(f"- screen validation ({r.encoder}, {r.role}): Spearman rho {r.spearman_rho:.2f}, one-sided p "
              f"{r.perm_p_one_sided:.4f}; missed: {r.missed_by_metadata_screen} -> {r.reading}")
if (OUT_ROOT / 'decoding' / 'acquisition_decoding.csv').exists():
    dc = pd.read_csv(OUT_ROOT / 'decoding' / 'acquisition_decoding.csv')
    if 'pooled_auc' in dc:
        g = dc[dc.input == 'background'].dropna(subset=['pooled_auc'])
        print(f"\nEXPERIMENT C (label-free), background only: pooled AUC {g.pooled_auc.min():.2f}-{g.pooled_auc.max():.2f}, "
              f"within collection {g.within_collection_auc.min():.2f}-{g.within_collection_auc.max():.2f}, "
              f"shuffled targets {g.shuffled_target_auc.min():.2f}-{g.shuffled_target_auc.max():.2f}")
print('\nNext: review the saved results and run log before writing up findings.')